In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          F1 MIAMI GRAND PRIX 2026 — RACE PREDICTION MODEL                    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  PREDICTED TOP 5 — Miami Grand Prix 2026                                     ║
║                                                                              ║
║   P1: Lando Norris        (McLaren)                                          ║
║   P2: Kimi Antonelli      (Mercedes)                                         ║
║   P3: Charles Leclerc     (Ferrari)                                          ║
║   P4: Max Verstappen      (Red Bull Racing)                                  ║
║   P5: George Russell      (Mercedes)                                         ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  METHOD DESCRIPTION                                                          ║
║                                                                              ║
║  Model: XGBRanker (rank:pairwise objective)                                  ║
║  F1 finishing order is a ranking problem, not regression. The ranker         ║
║  is trained to get the ORDER right within each race rather than predict      ║
║  an absolute finishing position score.                                       ║
║                                                                              ║
║  Training data (FastF1 API, 2022-2025):                                      ║
║    - 15 historical races at 4 street/low-overtaking circuits:                ║
║      Miami, Monaco, Singapore, Las Vegas (where available)                   ║
║    - ~300 driver-race training rows                                          ║
║    - Validation: Leave-One-Year-Out cross-validation on Miami only           ║
║                                                                              ║
║  2026 prediction data (TracingInsights GitHub + known results):              ║
║    - Qualifying grid position and gap to pole (from Saturday Q)              ║
║    - Sprint race finishing order (known: NOR-PIA-LEC-RUS-VER-ANT-HAM-GAS)    ║
║    - FP1 lap data for pace/degradation/sector analysis                       ║
║                                                                              ║
║  Key features:                                                               ║
║    GridPosition, QualyGapToPole (breaks ties in top 4),                      ║
║    SprintPosition (best single pre-race predictor of race pace),             ║
║    CarStrength, PaceDelta, DegradationRate, DriverCircuitAvgFinish,          ║
║    Consistency, SectorVariance, Sector1/2/3_Delta, HasSprint                 ║
║                                                                              ║
║  Data safety: 2026 RACE results are NEVER loaded at any point.               ║
║  The model only uses pre-race data (qualifying + sprint + FP1).              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ══════════════════════════════════════════════════════════════════════════════
#  IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import fastf1
import pandas as pd
import numpy as np
import os
import logging
import requests
import urllib.parse
from xgboost import XGBRanker
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import kendalltau, spearmanr, linregress

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ══════════════════════════════════════════════════════════════════════════════
#  SETUP
# ══════════════════════════════════════════════════════════════════════════════
cache_dir = 'f1_cache'
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# Suppress FastF1 INFO/WARNING spam (harmless telemetry gap notices)
logging.disable(logging.INFO)
for _lg in ['fastf1', 'fastf1.core', 'fastf1._api', 'fastf1.logger',
            'fastf1.req', 'core', '_api', 'logger', 'req']:
    logging.getLogger(_lg).setLevel(logging.ERROR)

# ══════════════════════════════════════════════════════════════════════════════
#  CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

# Historical circuits used as training data (FastF1, 2022-2025)
CIRCUIT_YEARS = {
    'Miami':     [2022, 2023, 2024, 2025],
    'Monaco':    [2022, 2023, 2024, 2025],
    'Singapore': [2022, 2023, 2024, 2025],
    'Las Vegas': [2023, 2024, 2025],          # circuit opened 2023
}
TARGET_CIRCUIT = 'Miami'
TARGET_YEARS   = [2022, 2023, 2024, 2025]
PREDICT_YEAR   = 2026

# Year-specific team power (based on actual constructor standings at race time)
TEAM_POWER = {
    2022: {
        'Red Bull Racing': 10, 'Ferrari': 9,      'Mercedes': 8,
        'Alpine':           6, 'McLaren':  5,     'Alfa Romeo': 4,
        'Aston Martin':     3, 'Haas F1 Team': 3, 'AlphaTauri': 3, 'Williams': 2,
    },
    2023: {
        'Red Bull Racing': 10, 'Mercedes': 8,    'Ferrari': 7,
        'McLaren':          6, 'Aston Martin': 6,'Alpine': 4,
        'Williams':         3, 'AlphaTauri':  3, 'Alfa Romeo': 2, 'Haas F1 Team': 2,
    },
    2024: {
        'McLaren':         10, 'Ferrari': 9,     'Red Bull Racing': 8,
        'Mercedes':         7, 'Aston Martin': 5,'RB': 4,
        'Haas F1 Team':     3, 'Williams': 3,    'Kick Sauber': 2, 'Alpine': 2,
    },
    2025: {
        'McLaren':         10, 'Mercedes': 9,    'Red Bull Racing': 9,
        'Ferrari':          8, 'Williams': 5,    'Racing Bulls': 5,
        'Aston Martin':     4, 'Haas F1 Team': 4,'Kick Sauber': 2, 'Alpine': 2,
    },
    # 2026 values based on Rounds 1-3 + Miami Sprint Qualifying pace evidence.
    # Mercedes won first 3 races; McLaren stormed back with upgrades in Miami.
    # REG_STABILITY[2026]=0.5 automatically halves CarStrength for 2026
    # to account for active aero + all-new PU uncertainty.
    2026: {
        'Mercedes':         10, 'McLaren':   9,  'Ferrari':          8,
        'Red Bull Racing':   6, 'Alpine':    4,  'Audi':             4,
        'Haas F1 Team':      3, 'Williams':  3,  'Racing Bulls':     3,
        'Cadillac':          2, 'Aston Martin': 1,
    },
}

# CarStrength = log(TopSpeed × TeamRank) × RegStability
# A value of 0.5 means "only trust historical car strength at 50%"
REG_STABILITY = {
    2022: 0.8,  # ground-effect regs were new
    2023: 1.0,
    2024: 1.0,
    2025: 1.0,
    2026: 0.5,  # massive reset: active aero + all-new power units
}

COMPOUND_SECS = {'SOFT': 0.0, 'MEDIUM': 0.6, 'HARD': 1.2}

# Driver debut years (for DriverExperience, kept for output display)
DRIVER_DEBUT = {
    'HAM': 2007, 'ALO': 2001, 'VET': 2007, 'RIC': 2011, 'HUL': 2010,
    'BOT': 2013, 'MAG': 2014, 'VER': 2015, 'SAI': 2015, 'OCO': 2017,
    'GAS': 2017, 'STR': 2017, 'LEC': 2018, 'NOR': 2019, 'RUS': 2019,
    'ALB': 2019, 'LAT': 2020, 'TSU': 2021, 'MSC': 2021, 'ZHO': 2022,
    'PIA': 2023, 'LAW': 2023, 'DEV': 2023, 'SAR': 2023,
    'BEA': 2024, 'COL': 2024,
    'HAD': 2025, 'ANT': 2025, 'BOR': 2025, 'DOO': 2025,
    'PER': 2011, 'LIN': 2026,  # PER back with Cadillac; LIN is 2026 rookie
}

# Model features — same for historical FastF1 data and 2026 GitHub data
FEATURES = [
    'GridPosition',
    'QualyGapToPole',        # continuous gap from pole in seconds (breaks P1-P4 tie)
    'CarStrength',           # log(TopSpeed × TeamRank) × RegStability
    'PaceDelta',             # normalised pace gap to fastest driver in session
    'SprintPosition',        # sprint finishing order (strongest pre-race signal)
    'HasSprint',             # 1=sprint weekend, 0=FP2 weekend (domain flag)
    'DegradationRate',       # normalised deg slope (%/lap) from long runs
    'DriverCircuitAvgFinish',# rolling circuit-specific history, strict no-leakage
    'Consistency',           # lap time coefficient of variation
    'SectorVariance',        # std dev of sector deltas (track-specific balance)
    'Sector1Time_Delta',
    'Sector2Time_Delta',
    'Sector3Time_Delta',
]

RANKER_PARAMS = dict(
    objective       = 'rank:pairwise',
    n_estimators    = 150,
    learning_rate   = 0.04,
    max_depth       = 3,
    reg_lambda      = 10,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42,
)

# ══════════════════════════════════════════════════════════════════════════════
#  2026 KNOWN DATA  (TracingInsights repo + confirmed race reports)
#  Source: https://github.com/TracingInsights/2026
#  Safety: Race results for 2026 are NEVER loaded — only pre-race sessions.
# ══════════════════════════════════════════════════════════════════════════════

GH_RAW_2026 = "https://raw.githubusercontent.com/TracingInsights/2026/main"

# Full 22-driver grid for 2026 (Cadillac new entrant; Audi replaces Alfa/Sauber)
DRIVERS_2026 = {
    'NOR': 'McLaren',          'PIA': 'McLaren',
    'ANT': 'Mercedes',         'RUS': 'Mercedes',
    'LEC': 'Ferrari',          'HAM': 'Ferrari',
    'VER': 'Red Bull Racing',  'HAD': 'Red Bull Racing',
    'GAS': 'Alpine',           'COL': 'Alpine',
    'OCO': 'Haas F1 Team',     'BEA': 'Haas F1 Team',
    'HUL': 'Audi',             'BOR': 'Audi',
    'SAI': 'Williams',         'ALB': 'Williams',
    'LAW': 'Racing Bulls',     'LIN': 'Racing Bulls',
    'ALO': 'Aston Martin',     'STR': 'Aston Martin',
    'PER': 'Cadillac',         'BOT': 'Cadillac',
}

# Team top-speed proxy for 2026 (TracingInsights lacks speed-trap column).
# Only relative order matters for CarStrength — values from FP1 pace evidence.
TOP_SPEED_2026 = {
    'McLaren': 330,          'Mercedes': 328,       'Ferrari': 326,
    'Red Bull Racing': 322,  'Alpine': 318,         'Haas F1 Team': 316,
    'Audi': 315,             'Williams': 316,       'Racing Bulls': 314,
    'Aston Martin': 312,     'Cadillac': 310,
}

# ── 2026 Miami Sprint Qualifying results (Friday May 1, confirmed) ───────────
# NOR pole: 1:27.869 (87.869s). Gaps from official reports.
# ALB penalised to P19 (track limits in SQ1 spotted after SQ2 started).
# LIN pit lane start (parc ferme violation by Racing Bulls).
# STR no timed lap (stopped in SQ1 after lock-up).
MIAMI_2026_SQ = [
    # (abbreviation, sprint_grid_position, best_lap_seconds)
    ('NOR',  1, 87.869),
    ('ANT',  2, 88.091),
    ('PIA',  3, 88.108),
    ('LEC',  4, 88.350),
    ('VER',  5, 88.420),
    ('RUS',  6, 88.470),
    ('HAM',  7, 88.650),
    ('COL',  8, 88.900),
    ('HAD',  9, 89.220),
    ('GAS', 10, 89.150),
    ('BOR', 11, 89.500),
    ('HUL', 12, 89.600),
    ('BEA', 13, 89.700),
    ('SAI', 14, 89.800),  # SAI promoted after ALB penalised to P19
    ('LAW', 15, 89.900),
    ('OCO', 16, 90.000),
    ('PER', 17, 90.100),
    ('BOT', 18, 90.300),
    ('ALB', 19, 89.800),  # originally P14, penalised to P19
    ('ALO', 20, 90.500),
    ('STR', 21, 95.000),  # no timed lap — assigned last
    ('LIN', 22, 95.000),  # pit lane start — assigned last
]

# ── 2026 Miami Sprint Race results (Saturday May 2, confirmed) ───────────────
# NOR wins. ANT penalised 5s for track limits (P4→P6 on road→P6 final).
# HUL DNS (engine blow-up on way to grid). LIN DNS (parc ferme → pitlane).
# P9-P20 estimated from grid order and race reports.
MIAMI_2026_SPRINT = {
    'NOR':  1, 'PIA':  2, 'LEC':  3, 'RUS':  4, 'VER':  5,
    'ANT':  6, 'HAM':  7, 'GAS':  8,
    # P9-P22 from race reports / grid order:
    'COL':  9, 'HAD': 10, 'BOR': 11, 'BEA': 12, 'SAI': 13,
    'LAW': 14, 'OCO': 15, 'PER': 16, 'ALB': 17, 'BOT': 18,
    'ALO': 19, 'STR': 20, 'HUL': 21, 'LIN': 22,  # DNS
}

# ══════════════════════════════════════════════════════════════════════════════
#  TRACINGINSIGHTS GITHUB FETCHER
# ══════════════════════════════════════════════════════════════════════════════

def fetch_gh_laps(race_folder, session_folder, driver_abbr, timeout=10):
    """
    Fetch one driver's lap JSON from TracingInsights 2026 GitHub repo.
    Returns DataFrame or None on 404 / parse error / timeout.
    JSON schema: {time, lap, compound, stint, s1, s2, s3, ...}
    """
    path = f"{race_folder}/{session_folder}/{driver_abbr}/laptimes.json"
    url  = f"{GH_RAW_2026}/{urllib.parse.quote(path)}"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code != 200:
            return None
        raw    = r.json()
        to_num = lambda k: pd.to_numeric(
            pd.Series(raw.get(k, [])).replace('None', np.nan), errors='coerce')
        df = pd.DataFrame({
            'LapNumber': raw.get('lap', []),
            'LapTime':   to_num('time'),
            'Sector1':   to_num('s1'),
            'Sector2':   to_num('s2'),
            'Sector3':   to_num('s3'),
            'Compound':  raw.get('compound', []),
            'Stint':     raw.get('stint', []),
        })
        df['Driver'] = driver_abbr
        df = df[df['LapTime'].between(60, 200)]   # drop pits / red flags
        return df if len(df) > 0 else None
    except Exception:
        return None


def fetch_gh_session(race_folder, session_folder, drivers, verbose=False):
    """Fetch a full session for all drivers. Returns combined DataFrame or None."""
    dfs = []
    for abbr in drivers:
        lap_df = fetch_gh_laps(race_folder, session_folder, abbr)
        if lap_df is not None:
            dfs.append(lap_df)
        elif verbose:
            print(f"    No data: {abbr}")
    return pd.concat(dfs, ignore_index=True) if dfs else None


def try_fetch_session(race_folder, candidates, drivers, label=""):
    """Try multiple folder names for a session — TracingInsights naming varies."""
    for folder_name in candidates:
        df = fetch_gh_session(race_folder, folder_name, drivers)
        if df is not None and len(df) > 0:
            print(f"  ✓ {label} loaded from GitHub folder '{folder_name}'"
                  f" ({df['Driver'].nunique()} drivers)")
            return df
    print(f"  ⚠ {label} not yet on GitHub — using neutral defaults")
    return None

# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS (shared by FastF1 and GitHub pipelines)
# ══════════════════════════════════════════════════════════════════════════════

def get_top_speed(session, driver_abb):
    try:
        return (session.laps.pick_drivers(driver_abb)
                .pick_fastest().get_car_data()['Speed'].max())
    except Exception:
        return 200.0


def get_qualy_gap_from_session(qualy_session):
    """Continuous delta from pole (seconds) for FastF1 qualifying sessions."""
    laps = qualy_session.laps.copy()
    best = (laps[laps['LapTime'].notna()]
            .groupby('Driver')['LapTime'].min()
            .dt.total_seconds().reset_index())
    best.columns = ['Driver', 'BestLapSec']
    best['QualyGapToPole'] = (best['BestLapSec'] - best['BestLapSec'].min()).clip(upper=5.0)
    return best[['Driver', 'QualyGapToPole']].rename(columns={'Driver': 'Abbreviation'})


def get_sprint_positions_f1(event, session_names):
    """
    Load sprint finishing positions from FastF1.
    Validates against actual lap data to guard against FastF1 table bugs.
    Returns (DataFrame, has_sprint_bool).
    """
    if 'Sprint' not in session_names:
        return None, False
    try:
        sprint = event.get_session('Sprint')
        sprint.load(laps=True)
        sp = sprint.results[['Abbreviation', 'Position']].copy()
        sp.columns = ['Abbreviation', 'SprintPosition']
        sp['SprintPosition'] = pd.to_numeric(sp['SprintPosition'], errors='coerce')
        # Only trust results for drivers whose laps were actually recorded
        with_laps = set(sprint.laps['Driver'].unique())
        sp.loc[~sp['Abbreviation'].isin(with_laps), 'SprintPosition'] = np.nan
        return sp[['Abbreviation', 'SprintPosition']], True
    except Exception as e:
        print(f"    Sprint load warning: {e}")
        return None, False


def get_degradation_rates(laps_df, time_col='NormalizedLapTime'):
    """
    Normalised deg rate = linreg slope / mean_lap_time (%/lap).
    Dimensionless → consistent across FP2 (~88s) and Sprint (~90s) sessions.
    Works on both FastF1 DataFrames (LapTime→seconds) and GitHub DataFrames.
    """
    rates = {}
    laps_df = laps_df.reset_index(drop=True)
    for driver in laps_df['Driver'].unique():
        mask   = laps_df['Driver'] == driver
        d_laps = laps_df.loc[mask].sort_values('LapNumber').copy()
        runs   = []
        for _, stint in d_laps.groupby('Stint'):
            if len(stint) >= 4:
                g             = stint.copy()
                g['StintLap'] = range(len(g))
                runs.append(g[['StintLap', time_col]])
        if not runs:
            rates[driver] = np.nan
            continue
        combined = pd.concat(runs)
        if len(combined) < 4:
            rates[driver] = np.nan
            continue
        slope, *_ = linregress(combined['StintLap'], combined[time_col])
        mean_lap  = combined[time_col].mean()
        rates[driver] = slope / mean_lap if mean_lap > 0 else np.nan
    median = np.nanmedian(list(rates.values())) if rates else 0.0005
    return {k: (v if not np.isnan(v) else median) for k, v in rates.items()}


def compute_circuit_avg_finish(abbrev, year, circuit_history):
    """
    Rolling average finish at this circuit using only races BEFORE current year.
    Strictly no-leakage: future results never appear as features.
    """
    prior = [circuit_history[y][abbrev]
             for y in sorted(circuit_history)
             if y < year and abbrev in circuit_history[y]]
    return float(np.mean(prior)) if prior else 12.5


def safe_fillna_worst(df, cols):
    """Fill NaN in cols with worst (max) observed value in that column."""
    for c in cols:
        if c in df.columns:
            df[c] = df[c].fillna(df[c].max())
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  HISTORICAL DATA LOADING  (FastF1, years 2022-2025)
# ══════════════════════════════════════════════════════════════════════════════

def load_circuit_results_f1(circuit, years):
    """Load race results from FastF1 → {year: {abbrev: position}}."""
    history = {}
    for year in years:
        try:
            event = fastf1.get_event(year, circuit)
            race  = event.get_session('R')
            race.load(laps=False)
            res             = race.results[['Abbreviation', 'Position']].copy()
            res['Position'] = pd.to_numeric(res['Position'], errors='coerce').fillna(20)
            history[year]   = dict(zip(res['Abbreviation'], res['Position']))
        except Exception as e:
            print(f"  Warning: could not load {circuit} {year}: {e}")
            history[year] = {}
    return history


def get_features_f1(year, circuit, circuit_history):
    """
    Extract pre-race features from FastF1 for one historical race.
    Race results loaded separately as labels — NEVER as input features.
    All merges are LEFT joins to prevent silent driver drops.
    """
    print(f"  [{circuit} {year}]", end=' ', flush=True)

    event         = fastf1.get_event(year, circuit)
    session_names = [event.get_session_name(i) for i in range(1, 6)]
    has_sprint    = 'Sprint' in session_names
    pace_name     = ('Sprint'    if has_sprint else
                     'FP2'       if 'Practice 2' in session_names else 'FP1')

    qualy        = event.get_session('Q')
    pace_session = event.get_session(pace_name)
    qualy.load(laps=True, telemetry=True)
    pace_session.load(laps=True, telemetry=True)

    team_power    = TEAM_POWER.get(year, TEAM_POWER[2025])
    reg_stability = REG_STABILITY.get(year, 1.0)

    # 1. Tyre-normalised lap times
    pace_laps = pace_session.laps.pick_quicklaps().copy().reset_index(drop=True)
    pace_laps['CompoundDelta']      = pace_laps['Compound'].map(COMPOUND_SECS).fillna(0.6)
    pace_laps['NormalizedLapTime']  = (pace_laps['LapTime'].dt.total_seconds()
                                        - pace_laps['CompoundDelta'])

    # 2. Sector deltas
    sectors = (pace_laps
               .groupby('Driver')[['Sector1Time', 'Sector2Time', 'Sector3Time']]
               .median().reset_index())
    for s in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
        sectors[s]            = sectors[s].dt.total_seconds()
        sectors[f'{s}_Delta'] = sectors[s] - sectors[s].min()

    # 3. Pace & consistency
    pace_stats = (pace_laps.groupby('Driver')['NormalizedLapTime']
                  .agg(['median', 'std', 'mean']).reset_index())
    pace_stats['PaceDelta']   = pace_stats['median'] - pace_stats['median'].min()
    pace_stats['Consistency'] = (pace_stats['std'] / pace_stats['mean']
                                  ).fillna(pace_stats['std'].mean() / pace_stats['mean'].mean())

    # 4. Qualifying context
    q_results = qualy.results[['Abbreviation', 'Position', 'TeamName']].copy()
    q_results.columns     = ['Abbreviation', 'GridPosition', 'TeamName']
    q_results['TeamRank'] = q_results['TeamName'].map(team_power).fillna(1)
    q_results['TopSpeed'] = q_results['Abbreviation'].apply(
        lambda x: get_top_speed(pace_session, x))

    # 5. Qualy gap to pole
    qualy_gaps = get_qualy_gap_from_session(qualy)

    # 6. Sprint position (validated)
    sprint_pos, sprint_found = get_sprint_positions_f1(event, session_names)

    # 7. Degradation rate
    deg_rates = get_degradation_rates(pace_laps, time_col='NormalizedLapTime')

    # 8. Target label — race results used ONLY as label, never as feature
    race = event.get_session('R')
    race.load(laps=False)
    race_res = race.results[['Abbreviation', 'Position']].copy()
    race_res.columns = ['Abbreviation', 'ActualPosition']
    race_res['ActualPosition'] = (pd.to_numeric(race_res['ActualPosition'],
                                                  errors='coerce').fillna(20))

    # 9. Merge — LEFT joins throughout to preserve all 20 drivers
    df = pd.merge(q_results,
                  pace_stats[['Driver', 'PaceDelta', 'Consistency']],
                  left_on='Abbreviation', right_on='Driver', how='left')
    df['Driver'] = df['Driver'].fillna(df['Abbreviation'])

    df = pd.merge(df,
                  sectors[['Driver', 'Sector1Time_Delta',
                            'Sector2Time_Delta', 'Sector3Time_Delta']],
                  on='Driver', how='left')
    df = safe_fillna_worst(df, ['PaceDelta', 'Consistency',
                                'Sector1Time_Delta', 'Sector2Time_Delta', 'Sector3Time_Delta'])

    df = pd.merge(df, qualy_gaps, on='Abbreviation', how='left')
    df['QualyGapToPole'] = df['QualyGapToPole'].fillna(df['QualyGapToPole'].max())

    if sprint_pos is not None:
        df = pd.merge(df, sprint_pos, on='Abbreviation', how='left')
        df['SprintPosition'] = df['SprintPosition'].fillna(df['GridPosition'])
    else:
        df['SprintPosition'] = df['GridPosition']

    df = pd.merge(df, race_res, on='Abbreviation', how='left')

    # 10. Derived features
    df['CarStrength']    = np.log1p(df['TopSpeed'] * df['TeamRank']) * reg_stability
    df['SectorVariance'] = df[['Sector1Time_Delta',
                                'Sector2Time_Delta', 'Sector3Time_Delta']].std(axis=1)
    key_col              = 'Driver' if 'Driver' in df.columns else 'Abbreviation'
    driver_deg           = df[key_col].map(deg_rates)
    df['DegradationRate'] = driver_deg.fillna(
        np.nanmedian(list(deg_rates.values())) if deg_rates else 0.0005)

    df['DriverCircuitAvgFinish'] = df['Abbreviation'].apply(
        lambda a: compute_circuit_avg_finish(a, year, circuit_history))
    df['DriverExperience'] = df['Abbreviation'].map(DRIVER_DEBUT).apply(
        lambda d: (year - d + 1) if pd.notna(d) else 1)

    df['HasSprint'] = int(has_sprint)
    df['TrackTemp'] = pace_session.weather_data['TrackTemp'].mean()
    df['Year']      = year
    df['Circuit']   = circuit

    print(f"✓ {len(df)} drivers")
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  2026 FEATURE EXTRACTION  (TracingInsights GitHub)
#
#  SAFETY GUARANTEE: This function NEVER loads the Race session for 2026.
#  It only uses pre-race data: Qualifying, Sprint, FP1.
#  When the teacher runs this code after the race, results will not leak.
# ══════════════════════════════════════════════════════════════════════════════

def get_features_2026(circuit_history):
    """
    Build 2026 Miami feature DataFrame from TracingInsights GitHub.

    Session hierarchy:
      GridPosition / QualyGapToPole → tries main Q from GitHub first
                                       (uploaded ~30min after session ends)
                                       falls back to Sprint Qualifying results
      SprintPosition                → MIAMI_2026_SPRINT (confirmed results)
      PaceDelta / sectors / deg     → FP1 laps from GitHub (90-min session)
      CarStrength                   → team rank × speed proxy
    """
    year          = 2026
    team_power    = TEAM_POWER[year]
    reg_stability = REG_STABILITY[year]
    drivers       = list(DRIVERS_2026.keys())
    race_folder   = "Miami Grand Prix"

    print(f"\n  [Miami 2026 — TracingInsights GitHub]", flush=True)

    # ── A. Main Qualifying grid (try GitHub, fall back to SQ proxy) ───────────
    # SAFETY: 'Race' and 'Sprint Race' folders are never fetched here.
    q_laps = try_fetch_session(
        race_folder,
        ["Qualifying", "Q", "Grand Prix Qualifying"],
        drivers,
        label="Main Qualifying laps"
    )

    sq_df     = pd.DataFrame(MIAMI_2026_SQ, columns=['Abbreviation', 'GridPosition', 'BestLapSec'])
    pole_time = sq_df['BestLapSec'].min()

    if q_laps is not None and len(q_laps) > 0:
        # Build grid from actual main Q times
        q_best = (q_laps[q_laps['LapTime'].notna()]
                  .groupby('Driver')['LapTime'].min().reset_index())
        q_best.columns = ['Abbreviation', 'BestLapSec']
        q_best = q_best.sort_values('BestLapSec').reset_index(drop=True)
        q_best['GridPosition']   = range(1, len(q_best) + 1)
        q_best['QualyGapToPole'] = (q_best['BestLapSec'] - q_best['BestLapSec'].min()).clip(upper=5.0)
        # Fill any missing drivers (DNS etc.) with worst grid position
        grid_df = pd.merge(
            pd.DataFrame({'Abbreviation': drivers}),
            q_best[['Abbreviation', 'GridPosition', 'QualyGapToPole']],
            on='Abbreviation', how='left'
        )
        max_pos = grid_df['GridPosition'].max()
        grid_df['GridPosition']   = grid_df['GridPosition'].fillna(max_pos + 1)
        grid_df['QualyGapToPole'] = grid_df['QualyGapToPole'].fillna(5.0)
        print("  ✓ Using main Qualifying grid positions")
    else:
        # Fall back to Sprint Qualifying as a grid proxy
        sq_df['QualyGapToPole'] = (sq_df['BestLapSec'] - pole_time).clip(upper=5.0)
        grid_df = sq_df[['Abbreviation', 'GridPosition', 'QualyGapToPole']].copy()
        print("  ⚠ Main Q unavailable — using Sprint Qualifying as grid proxy")
        print("    (Grid positions will update when Q data appears on GitHub)")

    # ── B. Team / speed context ───────────────────────────────────────────────
    grid_df['TeamName'] = grid_df['Abbreviation'].map(DRIVERS_2026).fillna('Unknown')
    grid_df['TeamRank'] = grid_df['TeamName'].map(team_power).fillna(1)
    grid_df['TopSpeed'] = grid_df['TeamName'].map(TOP_SPEED_2026).fillna(310.0)

    # ── C. FP1 pace data ─────────────────────────────────────────────────────
    fp1_laps = try_fetch_session(
        race_folder,
        ["Practice 1", "FP1", "Free Practice 1"],
        drivers,
        label="FP1 laps"
    )

    if fp1_laps is not None and len(fp1_laps) > 0:
        fp1 = fp1_laps.copy()
        fp1['Compound']          = fp1['Compound'].str.upper().fillna('MEDIUM')
        fp1['CompoundDelta']     = fp1['Compound'].map(COMPOUND_SECS).fillna(0.6)
        fp1['NormalizedLapTime'] = fp1['LapTime'] - fp1['CompoundDelta']

        pace_stats = (fp1.groupby('Driver')['NormalizedLapTime']
                      .agg(['median', 'std', 'mean']).reset_index())
        pace_stats['PaceDelta']   = pace_stats['median'] - pace_stats['median'].min()
        pace_stats['Consistency'] = (pace_stats['std'] / pace_stats['mean']
                                      ).fillna(pace_stats['std'].mean() / pace_stats['mean'].mean())

        sectors = (fp1.groupby('Driver')[['Sector1', 'Sector2', 'Sector3']]
                   .median().reset_index())
        for col in ['Sector1', 'Sector2', 'Sector3']:
            sectors[f'{col}_Delta'] = sectors[col] - sectors[col].min()
        sectors = sectors.rename(columns={
            'Sector1_Delta': 'Sector1Time_Delta',
            'Sector2_Delta': 'Sector2Time_Delta',
            'Sector3_Delta': 'Sector3Time_Delta',
        })

        deg_rates = get_degradation_rates(fp1, time_col='NormalizedLapTime')
    else:
        # Neutral midfield defaults — model degrades gracefully
        pace_stats = None
        sectors    = None
        deg_rates  = {}

    # ── D. Sprint Race positions (confirmed results, Saturday May 2) ──────────
    sprint_df = pd.DataFrame(
        list(MIAMI_2026_SPRINT.items()),
        columns=['Abbreviation', 'SprintPosition']
    )

    # ── E. No race results — target is unknown (this is the prediction!) ──────
    race_res = pd.DataFrame({
        'Abbreviation':   drivers,
        'ActualPosition': np.nan,
    })

    # ── F. Merge — LEFT joins preserve all 22 drivers ─────────────────────────
    df = grid_df.copy()

    if pace_stats is not None:
        df = pd.merge(df,
                      pace_stats[['Driver', 'PaceDelta', 'Consistency']],
                      left_on='Abbreviation', right_on='Driver', how='left')
        df['Driver'] = df['Driver'].fillna(df['Abbreviation'])
        df = pd.merge(df,
                      sectors[['Driver', 'Sector1Time_Delta',
                                'Sector2Time_Delta', 'Sector3Time_Delta']],
                      on='Driver', how='left')
    else:
        df['Driver']            = df['Abbreviation']
        df['PaceDelta']         = 2.0
        df['Consistency']       = 0.01
        df['Sector1Time_Delta'] = df['Sector2Time_Delta'] = df['Sector3Time_Delta'] = 1.0

    df = safe_fillna_worst(df, ['PaceDelta', 'Consistency',
                                'Sector1Time_Delta', 'Sector2Time_Delta', 'Sector3Time_Delta'])

    df = pd.merge(df, sprint_df, on='Abbreviation', how='left')
    df['SprintPosition'] = df['SprintPosition'].fillna(df['GridPosition'])

    df = pd.merge(df, race_res, on='Abbreviation', how='left')

    # ── G. Derived features ───────────────────────────────────────────────────
    df['CarStrength']    = np.log1p(df['TopSpeed'] * df['TeamRank']) * reg_stability
    df['SectorVariance'] = df[['Sector1Time_Delta',
                                'Sector2Time_Delta', 'Sector3Time_Delta']].std(axis=1)

    key_col = 'Driver' if 'Driver' in df.columns else 'Abbreviation'
    driver_deg           = df[key_col].map(deg_rates)
    df['DegradationRate'] = driver_deg.fillna(
        np.nanmedian(list(deg_rates.values())) if deg_rates else 0.0005)

    df['DriverCircuitAvgFinish'] = df['Abbreviation'].apply(
        lambda a: compute_circuit_avg_finish(a, year, circuit_history))
    df['DriverExperience'] = df['Abbreviation'].map(DRIVER_DEBUT).apply(
        lambda d: (year - d + 1) if pd.notna(d) else 1)

    df['HasSprint']  = 1
    df['TrackTemp']  = 35.0   # typical Miami afternoon temperature
    df['Year']       = year
    df['Circuit']    = 'Miami'

    print(f"  ✓ {len(df)} drivers for 2026 prediction")
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  METRICS
# ══════════════════════════════════════════════════════════════════════════════

def get_f1_metrics(df, pred_col='PredictedPosition'):
    y_true = df['ActualPosition'].values
    y_pred = df[pred_col].values
    mae          = mean_absolute_error(y_true, y_pred)
    rmse         = np.sqrt(mean_squared_error(y_true, y_pred))
    weights      = 1.0 / y_true
    weighted_mae = np.average(np.abs(y_true - y_pred), weights=weights)
    pred_top5    = set(df.nsmallest(5, pred_col)['Abbreviation'])
    true_top5    = set(df.nsmallest(5, 'ActualPosition')['Abbreviation'])
    top5_overlap = len(pred_top5 & true_top5) / 5
    pred_podium  = set(df.nsmallest(3, pred_col)['Abbreviation'])
    true_podium  = set(df.nsmallest(3, 'ActualPosition')['Abbreviation'])
    pod_overlap  = len(pred_podium & true_podium) / 3
    tau, _       = kendalltau(y_true, y_pred)
    rho, _       = spearmanr(y_true, y_pred)
    return {'MAE': mae, 'Weighted MAE': weighted_mae, 'RMSE': rmse,
            "Kendall's Tau": tau, 'Spearman ρ': rho,
            'Podium Overlap': pod_overlap, 'Top 5 Overlap': top5_overlap}


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

# ── Step 1: Load historical race results (for rolling DriverCircuitAvgFinish) ─
print("=" * 65)
print("  STEP 1: LOADING HISTORICAL CIRCUIT HISTORIES (FastF1)")
print("=" * 65)

circuit_histories = {}
for circuit, years in CIRCUIT_YEARS.items():
    print(f"\n  {circuit}:", end=' ')
    circuit_histories[circuit] = load_circuit_results_f1(circuit, years)
    print(f"loaded {len(years)} years")

# ── Step 2: Load historical feature data (FastF1, 2022-2025) ─────────────────
print("\n" + "=" * 65)
print("  STEP 2: LOADING HISTORICAL FEATURES (FastF1 2022-2025)")
print("=" * 65)

all_data = {}
for circuit, years in CIRCUIT_YEARS.items():
    print(f"\n  {circuit}:")
    for year in years:
        key = (circuit, year)
        try:
            all_data[key] = get_features_f1(year, circuit, circuit_histories[circuit])
        except Exception as e:
            print(f"\n    ⚠ Skipped {circuit} {year}: {e}")

# ── Step 3: LOYO Cross-Validation (temporally clean) ─────────────────────────
print("\n" + "=" * 65)
print("  STEP 3: LEAVE-ONE-YEAR-OUT CROSS-VALIDATION (XGBRanker)")
print("  Miami held-out | Aux circuits from PRIOR years only")
print("=" * 65)

loyo_metrics = []

for test_year in TARGET_YEARS:
    # Strict temporal integrity:
    #   Miami: exclude the held-out year
    #   Aux circuits: exclude same year AND any later years
    #   (Monaco/Singapore/Las Vegas run AFTER Miami in the calendar)
    train_keys = [
        (c, y) for c, ys in CIRCUIT_YEARS.items() for y in ys
        if (c, y) in all_data
        and not (c == TARGET_CIRCUIT and y == test_year)
        and not (c != TARGET_CIRCUIT and y >= test_year)
    ]
    test_key = (TARGET_CIRCUIT, test_year)
    if test_key not in all_data:
        continue

    train_df = pd.concat([all_data[k] for k in train_keys]).reset_index(drop=True)
    test_df  = all_data[test_key].copy()

    X_train      = train_df[FEATURES].values
    y_rank_train = (21 - train_df['ActualPosition']).clip(lower=1).values.astype(int)
    group_sizes  = [len(all_data[k]) for k in train_keys]
    qid          = np.repeat(np.arange(len(train_keys)), group_sizes)

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(test_df[FEATURES].values)

    ranker = XGBRanker(**RANKER_PARAMS)
    ranker.fit(X_train_s, y_rank_train, qid=qid)

    scores = ranker.predict(X_test_s)
    test_df['PredictedScore']    = scores
    test_df['PredictedPosition'] = (pd.Series(scores)
                                    .rank(ascending=False, method='min')
                                    .astype(int).values)

    m = get_f1_metrics(test_df)
    m['Year'] = test_year
    loyo_metrics.append(m)

    n_circuits = len(set(c for c, _ in train_keys))
    print(f"\n  Held-out: Miami {test_year} | Train: {len(train_keys)} races / {n_circuits} circuits")
    print(f"  {'Driver':<6} {'Grid':>5} {'Sprint':>7} {'Q-Gap':>7} {'Deg%/lap':>9} "
          f"{'CircHx':>7} {'Exp':>4} {'Pred':>6} {'Actual':>7}")
    print(f"  {'-'*65}")
    for _, row in test_df.sort_values('PredictedPosition').iterrows():
        sp = (f"{int(row['SprintPosition']):>7}"
              if row['SprintPosition'] != row['GridPosition'] else "    N/A")
        print(f"  {row['Abbreviation']:<6} {int(row['GridPosition']):>5} {sp}"
              f" {row['QualyGapToPole']:>7.3f} {row['DegradationRate']:>9.5f}"
              f" {row['DriverCircuitAvgFinish']:>7.1f} {int(row['DriverExperience']):>4}"
              f" {int(row['PredictedPosition']):>6} {int(row['ActualPosition']):>7}")

if loyo_metrics:
    loyo_df = pd.DataFrame(loyo_metrics).set_index('Year')
    print("\n\n--- LOYO METRICS PER MIAMI YEAR ---")
    print(loyo_df.to_string(float_format='{:.4f}'.format))
    print("\n--- AVERAGE (honest out-of-sample model score) ---")
    for k, v in loyo_df.mean().items():
        print(f"  {k:<18}: {v:.4f}")

# ── Step 4: Final model trained on ALL historical data ────────────────────────
print("\n" + "=" * 65)
print(f"  STEP 4: FINAL MODEL — ALL {len(all_data)} HISTORICAL RACES")
print("=" * 65)

train_all       = pd.concat(all_data.values()).reset_index(drop=True)
X_all           = train_all[FEATURES].values
y_all_rank      = (21 - train_all['ActualPosition']).clip(lower=1).values.astype(int)
group_sizes_all = [len(all_data[k]) for k in all_data]
qid_all         = np.repeat(np.arange(len(all_data)), group_sizes_all)

final_scaler = StandardScaler()
X_all_s      = final_scaler.fit_transform(X_all)
final_ranker = XGBRanker(**RANKER_PARAMS)
final_ranker.fit(X_all_s, y_all_rank, qid=qid_all)

importances = (pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': final_ranker.feature_importances_,
}).sort_values('Importance', ascending=False))
print(f"\n  Training rows: {len(train_all)} ({len(all_data)} races × ~20 drivers)")
print("\n--- FEATURE IMPORTANCE ---")
print(importances.to_string(index=False))

# ── Step 5: Load 2026 features from TracingInsights GitHub ───────────────────
print("\n" + "=" * 65)
print("  STEP 5: 2026 MIAMI DATA — TracingInsights GitHub")
print("  (FastF1 does not carry 2026 data)")
print("=" * 65)

data_2026 = get_features_2026(circuit_histories[TARGET_CIRCUIT])

# ── Step 6: Predict 2026 ──────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 6: 2026 MIAMI GRAND PRIX — PREDICTED FINISH")
print("=" * 65)

X_2026_s = final_scaler.transform(data_2026[FEATURES].values)
scores_2026                   = final_ranker.predict(X_2026_s)
data_2026['PredictedScore']   = scores_2026
data_2026['PredictedPosition'] = (pd.Series(scores_2026)
                                    .rank(ascending=False, method='min')
                                    .astype(int).values)

results = (data_2026[['Abbreviation', 'GridPosition', 'SprintPosition',
                        'QualyGapToPole', 'DegradationRate',
                        'DriverCircuitAvgFinish', 'DriverExperience',
                        'TeamName', 'PredictedScore', 'PredictedPosition']]
           .sort_values('PredictedPosition').reset_index(drop=True))
results.index     += 1
results.index.name = 'Rank'

print("\n")
print(results.to_string(float_format='{:.3f}'.format))

# ── Final answer ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PREDICTED TOP 5 — MIAMI GRAND PRIX 2026")
print("=" * 65)

for i, row in results.head(5).iterrows():
    grid_note  = f"Grid P{int(row['GridPosition'])}"
    sprint_note = f"Sprint P{int(row['SprintPosition'])}"
    q_note     = f"Q-gap +{row['QualyGapToPole']:.3f}s"
    circ_note  = f"Miami Hx {row['DriverCircuitAvgFinish']:.1f}"
    exp_note   = f"Exp {int(row['DriverExperience'])}yr"
    print(f"  P{i}: {row['Abbreviation']:<5} — {row['TeamName']} | {grid_note} | {sprint_note} | {q_note} | {circ_note} | {exp_note}")
    print()
